<a href="https://colab.research.google.com/github/Hamdi-Jarban/SystemCRAFTCustomCRNN/blob/main/thai_ancient_ocr_crnn_curriculum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ نظام OCR للغة التايلاندية الأثرية (Ancient Thai OCR)
## بنية Custom Thai CRNN (CNN + BiLSTM + CTC) مبنية بالكامل من الصفر — باستخدام PyTorch
### استراتيجية Curriculum Learning عبر 5 مراحل تصاعدية الصعوبة

يحتوي هذا الدفتر (Notebook) على خط أنابيب متكامل (End-to-End Pipeline) جاهز للتشغيل مباشرة على Google Colab:

1. **إعداد البيئة** والربط بـ Google Drive.
2. **توليد بيانات تدريب تدرجية** (5 مراحل صعوبة) باستخدام مكتبة `trdg`.
3. **بناء معمارية CRNN مخصصة بالكامل** (بدون أي أوزان مدربة مسبقاً).
4. **تدريب تكاملي** ينتقل تلقائياً بين المراحل الخمس مع دعم الاستئناف التلقائي (Auto-Resume).
5. **الاستدلال وتقييم الأداء** عبر مقياس Character Error Rate (CER).

> ⚠️ **قبل التشغيل:** تأكد من تفعيل GPU من القائمة `Runtime > Change runtime type > GPU`، وارفع صور الخلفيات الأثرية الحقيقية إلى المجلد `/content/drive/MyDrive/Thai_OCR_Project/backgrounds` قبل الوصول إلى المرحلة الخامسة من التدريب.


In [ ]:
# ---- ربط Google Drive ----
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# الخلية 1: إعداد البيئة، الربط بـ Google Drive، وتجهيز المجلدات
# ============================================================

import os
import sys
import subprocess

# ---- تثبيت المكتبات المطلوبة ----
def pip_install(packages):
    for pkg in packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install([
    "trdg==1.8.0",
    "editdistance",
    "tqdm",
    "Pillow",
    "requests",
])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---- التحقق من توفر GPU ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ الجهاز المستخدم: {DEVICE}")
if torch.cuda.is_available():
    print(f"✅ اسم كرت الشاشة: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ تحذير: لم يتم العثور على GPU. فعّله من Runtime > Change runtime type > GPU")


# ---- تجهيز هيكل المجلدات ----
PROJECT_ROOT     = "/content/drive/MyDrive/Thai_OCR_Project"
BACKGROUNDS_DIR  = os.path.join("/content/drive/MyDrive/thai_assets/backgrounds")
DATASET_DIR      = os.path.join(PROJECT_ROOT, "dataset")
CHECKPOINTS_DIR  = os.path.join(PROJECT_ROOT, "checkpoints")
LOGS_DIR         = os.path.join(PROJECT_ROOT, "logs")

STAGE_NAMES = {
    1: "stage1_clean",
    2: "stage2_skew_blur",
    3: "stage3_noise",
    4: "stage4_heavy_distortion",
    5: "stage5_real_background",
}

for folder in [PROJECT_ROOT, BACKGROUNDS_DIR, DATASET_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    os.makedirs(folder, exist_ok=True)

for stage_id, stage_name in STAGE_NAMES.items():
    os.makedirs(os.path.join(DATASET_DIR, stage_name, "images"), exist_ok=True)

print("\n✅ تم إنشاء هيكل المجلدات بنجاح داخل:", PROJECT_ROOT)
print("📂 تأكد من رفع صور الخلفيات الأثرية الحقيقية داخل المجلد:")
print("   ", BACKGROUNDS_DIR)

import glob
bg_images = glob.glob(os.path.join(BACKGROUNDS_DIR, "*"))
print(f"🖼️ عدد صور الخلفيات الموجودة حالياً: {len(bg_images)}")
if len(bg_images) == 0:
    print("⚠️ لم يتم العثور على أي خلفيات بعد. المرحلة 5 ستحتاج صوراً داخل هذا المجلد قبل تشغيلها.")

✅ الجهاز المستخدم: cuda
✅ اسم كرت الشاشة: Tesla T4

✅ تم إنشاء هيكل المجلدات بنجاح داخل: /content/drive/MyDrive/Thai_OCR_Project
📂 تأكد من رفع صور الخلفيات الأثرية الحقيقية داخل المجلد:
    /content/drive/MyDrive/thai_assets/backgrounds
🖼️ عدد صور الخلفيات الموجودة حالياً: 686


### 🧠 الخلفية النظرية (الخلية 1)

هذه الخلية تُهيّئ بيئة العمل بشكل **قابل لإعادة الإنتاج (Reproducible)**، وهي خطوة جوهرية في أي بحث علمي: تثبيت إصدارات محددة من المكتبات، التأكد من تفعيل GPU لتسريع عمليات الالتفاف (Convolutions) والتكرار (Recurrence)، وربط المشروع بمساحة تخزين دائمة على Google Drive.

**لماذا Google Drive تحديداً؟** بيئة Colab تُعيد تصفير موارد التخزين المحلي (`/content`) عند انتهاء الجلسة، بينما نموذجنا يحتاج للتدريب عبر 5 مراحل متتالية قد تستغرق ساعات. لذلك، كل البيانات المولّدة ونقاط الحفظ (Checkpoints) تُكتب مباشرة إلى Drive لضمان **استمرارية العمل (Persistence)** حتى لو انقطع الاتصال بالجلسة — وهو الأساس الذي تُبنى عليه آلية الاستئناف التلقائي (Auto-Resume) لاحقاً في الخلية 4.


In [ ]:
%pip install -q --no-cache-dir \
    trdg==1.8.0 \
    Pillow==12.3.0 \
    arabic-reshaper==3.0.1 \
    python-bidi==0.6.11 \
    diffimg==0.3.0 \
    wikipedia \
    opencv-python \
    requests \
    tqdm


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install diffimg==0.3.0 and trdg==1.8.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
# ============================================================
# الخلية 2: مولد البيانات التدرجي (Curriculum Data Generator)
# ============================================================

import os
import glob
import random
import requests
import trdg.utils as trdg_utils

from trdg.generators import GeneratorFromDict, GeneratorFromStrings
from PIL import Image, ImageFont

IMG_HEIGHT = 64
SAMPLES_PER_STAGE = 1000   # يمكن تقليلها (مثلاً 300) لتسريع العرض التوضيحي غداً
VAL_SAMPLES = 200
WORDS_PER_LINE_MIN = 2
WORDS_PER_LINE_MAX = 5

random.seed(42)

# ---------------------------------------------------------
# إصلاح توافق TRDG 1.8.0 مع Pillow الحديث
# ---------------------------------------------------------

def _trdg_get_text_width(image_font, text):
    """
    بديل getsize القديم لحساب عرض النص.
    """
    if hasattr(image_font, "getlength"):
        return round(image_font.getlength(text))

    left, top, right, bottom = image_font.getbbox(text)
    return right - left


def _trdg_get_text_height(image_font, text):
    """
    بديل getsize القديم لحساب ارتفاع النص.
    """
    left, top, right, bottom = image_font.getbbox(text)
    return bottom - top


trdg_utils.get_text_width = _trdg_get_text_width
trdg_utils.get_text_height = _trdg_get_text_height

print("✅ تم إصلاح توافق TRDG مع Pillow الحديث")


# ---------------------------------------------------------
# 1) استخدام الخطوط التايلاندية الموجودة في Google Drive
# ---------------------------------------------------------

THAI_FONTS_DIR = "/content/drive/MyDrive/thai_assets/fonts/thai"


def ensure_fallback_thai_font():
    """
    البحث عن ملفات الخطوط التايلاندية الصالحة داخل Google Drive.
    تعيد قائمة بمسارات ملفات الخطوط، وليس مسار المجلد.
    """
    if not os.path.isdir(THAI_FONTS_DIR):
        print(f"⚠️ مجلد الخطوط غير موجود: {THAI_FONTS_DIR}")
        return []

    candidate_fonts = []

    for pattern in ("*.ttf", "*.otf", "*.TTF", "*.OTF"):
        candidate_fonts.extend(
            glob.glob(
                os.path.join(THAI_FONTS_DIR, "**", pattern),
                recursive=True
            )
        )

    valid_fonts = []

    for font_path in sorted(set(candidate_fonts)):
        try:
            # اختبار فعلي للتأكد من أن الملف خط صالح
            ImageFont.truetype(font_path, 32)
            valid_fonts.append(font_path)
        except Exception:
            # تجاهل ملفات الخطوط التالفة أو غير المدعومة
            pass

    print(f"✅ تم العثور على {len(valid_fonts)} خط تايلاندي صالح.")

    return valid_fonts


THAI_CONSONANTS = list(
    "กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ"
)

THAI_VOWELS = list(
    "ะัาำิีึืุูเแโใไๅ"
)

THAI_TONE_MARKS = list(
    "่้๊๋"
)


def generate_pseudo_thai_word():
    length = random.randint(2, 6)
    chars = []

    for _ in range(length):
        chars.append(random.choice(THAI_CONSONANTS))

        if random.random() < 0.5:
            chars.append(random.choice(THAI_VOWELS))

        if random.random() < 0.15:
            chars.append(random.choice(THAI_TONE_MARKS))

    return "".join(chars)


def generate_fallback_lines(num_lines):
    lines = []

    for _ in range(num_lines):
        num_words = random.randint(
            WORDS_PER_LINE_MIN,
            WORDS_PER_LINE_MAX
        )

        words = [
            generate_pseudo_thai_word()
            for _ in range(num_words)
        ]

        lines.append(" ".join(words))

    return lines


print("🔎 التحقق من توفر قاموس وخطوط اللغة التايلاندية المدمجة في trdg...")

FALLBACK_FONTS = []

try:
    _probe_generator = GeneratorFromDict(
        language="th",
        count=1,
        size=IMG_HEIGHT
    )

    _probe_img, _probe_label = next(iter(_probe_generator))

    if _probe_img is None:
        raise RuntimeError("الصورة الناتجة فارغة (None)")

    USE_BUILTIN_THAI_DICT = True

    print(
        "✅ trdg يدعم اللغة التايلاندية مباشرة "
        "(قاموس + خطوط). سيتم استخدامها."
    )

except Exception as probe_error:
    USE_BUILTIN_THAI_DICT = False

    print(
        "⚠️ تعذر استخدام قاموس/خطوط trdg التايلاندية "
        f"المدمجة تلقائياً ({probe_error})."
    )

    print(
        "🔄 التحويل التلقائي لاستخدام قائمة كلمات تايلاندية "
        "اصطناعية + الخطوط الموجودة في Google Drive."
    )

    FALLBACK_FONTS = ensure_fallback_thai_font()

    if not FALLBACK_FONTS:
        raise FileNotFoundError(
            "❌ لم يتم العثور على أي ملف خط TTF/OTF صالح داخل:\n"
            f"{THAI_FONTS_DIR}"
        )


# ---------------------------------------------------------
# 2) دالة بناء المولّد حسب رقم المرحلة
#    إعدادات تشويه مختلفة لكل مرحلة
# ---------------------------------------------------------

def build_generator(stage_id, count):
    """
    stage 1: خلفية بيضاء نقية بدون أي تشويه
    stage 2: ميلان بسيط + ضبابية خفيفة جداً
    stage 3: ضوضاء بصرية + تشوه خفيف في الخطوط (Sine)
    stage 4: تشويه وضبابية وتآكل أقوى
    stage 5: خلفيات أثرية حقيقية من Google Drive
            + نفس قوة تشويه المرحلة 4
    """

    shared_kwargs = dict(
        count=count,
        size=IMG_HEIGHT,
        fit=True,
        margins=(5, 5, 5, 5),
    )

    if USE_BUILTIN_THAI_DICT:
        shared_kwargs.update(
            language="th",
            length=3,
            allow_variable=True
        )
    else:
        # تمرير قائمة ملفات الخطوط، وليس مجلد الخطوط
        shared_kwargs.update(
            fonts=FALLBACK_FONTS
        )

    if stage_id == 1:
        stage_kwargs = dict(
            background_type=1,
            skewing_angle=0,
            random_skew=False,
            blur=0,
            random_blur=False,
            distorsion_type=0,
            distorsion_orientation=0
        )

    elif stage_id == 2:
        stage_kwargs = dict(
            background_type=1,
            skewing_angle=3,
            random_skew=True,
            blur=1,
            random_blur=True,
            distorsion_type=0,
            distorsion_orientation=0
        )

    elif stage_id == 3:
        stage_kwargs = dict(
            background_type=0,
            skewing_angle=5,
            random_skew=True,
            blur=1,
            random_blur=True,
            distorsion_type=1,
            distorsion_orientation=0
        )

    elif stage_id == 4:
        stage_kwargs = dict(
            background_type=2,
            skewing_angle=8,
            random_skew=True,
            blur=2,
            random_blur=True,
            distorsion_type=3,
            distorsion_orientation=2
        )

    elif stage_id == 5:
        if len(bg_images) == 0:
            raise FileNotFoundError(
                f"❌ المرحلة 5 تتطلب صور خلفيات أثرية داخل:\n"
                f"{BACKGROUNDS_DIR}\n"
                "يرجى رفع الصور ثم إعادة تشغيل هذه الخلية."
            )

        stage_kwargs = dict(
            background_type=3,
            image_dir=BACKGROUNDS_DIR,
            skewing_angle=6,
            random_skew=True,
            blur=2,
            random_blur=True,
            distorsion_type=3,
            distorsion_orientation=2
        )

    else:
        raise ValueError(f"رقم مرحلة غير معروف: {stage_id}")

    all_kwargs = {
        **shared_kwargs,
        **stage_kwargs
    }

    if USE_BUILTIN_THAI_DICT:
        return GeneratorFromDict(**all_kwargs)

    else:
        fallback_lines = generate_fallback_lines(count)

        return GeneratorFromStrings(
            fallback_lines,
            **all_kwargs
        )


# ---------------------------------------------------------
# 3) توليد صور كل مرحلة + ملف train_list.txt الخاص بها
# ---------------------------------------------------------

def generate_stage_dataset(
    stage_id,
    num_samples,
    is_validation=False
):
    stage_name = (
        STAGE_NAMES[stage_id]
        if not is_validation
        else "validation"
    )

    stage_folder = os.path.join(
        DATASET_DIR,
        stage_name
    )

    images_folder = os.path.join(
        stage_folder,
        "images"
    )

    os.makedirs(images_folder, exist_ok=True)

    list_filename = (
        "val_list.txt"
        if is_validation
        else "train_list.txt"
    )

    list_path = os.path.join(
        stage_folder,
        list_filename
    )

    if os.path.exists(list_path):
        with open(list_path, encoding="utf-8") as existing_file:
            existing_lines = sum(
                1 for _ in existing_file
            )

        if existing_lines >= num_samples:
            print(
                f"⏭️ تخطي التوليد: بيانات '{stage_name}' "
                f"موجودة مسبقاً ({existing_lines} عينة)."
            )
            return list_path

    generator = build_generator(
        stage_id,
        num_samples
    )

    written = 0

    with open(list_path, "w", encoding="utf-8") as f:
        for idx, (img, label) in enumerate(generator):

            if (
                img is None
                or label is None
                or len(label.strip()) == 0
            ):
                continue

            img = img.convert("L")

            img_name = f"{stage_name}_{idx:05d}.jpg"

            img_path = os.path.join(
                images_folder,
                img_name
            )

            img.save(img_path)

            rel_path = os.path.join(
                stage_name,
                "images",
                img_name
            )

            f.write(
                f"{rel_path}\t{label}\n"
            )

            written += 1

            if written >= num_samples:
                break

    print(
        f"✅ '{stage_name}': تم توليد {written} صورة "
        f"→ {list_path}"
    )

    return list_path


# ---------------------------------------------------------
# 4) بدء توليد مراحل التعلم التدرجي
# ---------------------------------------------------------

print(
    "\n🚀 بدء توليد بيانات مراحل التعلم التدرجي "
    "(Curriculum Learning)...\n"
)

stage_list_paths = {}

for stage_id in range(1, 6):
    stage_list_paths[stage_id] = generate_stage_dataset(
        stage_id,
        SAMPLES_PER_STAGE
    )

val_source_stage = (
    5
    if len(bg_images) > 0
    else 4
)

val_list_path = generate_stage_dataset(
    val_source_stage,
    VAL_SAMPLES,
    is_validation=True
)

print(
    "\n✅ انتهى توليد جميع مراحل البيانات بنجاح."
)


ModuleNotFoundError: No module named 'trdg'

### 🧠 الخلفية النظرية (الخلية 2) — التعلم التدرجي Curriculum Learning

فكرة **Curriculum Learning** (Bengio et al., 2009) مستوحاة من الطريقة التي يتعلم بها الإنسان: نبدأ بالأمثلة **السهلة** ثم ننتقل تدريجياً إلى الأمثلة **الأصعب**، بدلاً من عرض جميع مستويات الصعوبة على الشبكة العصبية دفعة واحدة منذ البداية.

في مشروعنا، النصوص التايلاندية الأثرية الحقيقية **نادرة ومشوّهة بشدة** (تآكل، ضوضاء، خلفيات غير منتظمة)، مما يجعل التدريب المباشر عليها منذ اللحظة الأولى غير مستقر وقد يقود النموذج لتعلّم أنماط خاطئة أو عدم التقارب إطلاقاً. لذلك نبني 5 مراحل متصاعدة الصعوبة:

| المرحلة | الصعوبة | الهدف التعليمي للنموذج |
|---|---|---|
| 1 | خلفية بيضاء نظيفة | تعلّم الشكل الأساسي لحروف اللغة التايلاندية |
| 2 | ميلان + ضبابية خفيفة | التعامل مع اختلافات بسيطة في الوضوح والزاوية |
| 3 | ضوضاء + تشوه خفيف | مقاومة التشويش البصري (Noise Robustness) |
| 4 | تشويه وتآكل قوي | محاكاة التلف الفيزيائي للمخطوطات القديمة |
| 5 | خلفيات أثرية حقيقية | التكيّف النهائي (Domain Adaptation) مع البيانات الواقعية |

هذا التدرج يمنح الشبكة **أساساً متيناً** في المراحل الأولى (شكل الحروف نفسه) قبل إجبارها على حل مشكلتين معاً (قراءة الحرف + مقاومة التشويش)، مما يحسّن الاستقرار أثناء التدريب والتعميم (Generalization) النهائي على البيانات الأثرية الحقيقية.


In [ ]:
# ============================================================
# الخلية 3: معمارية النموذج المخصص CustomThaiCRNN (من الصفر بالكامل)
# ============================================================

import torch
import torch.nn as nn

# ---------------------------------------------------------
# 1) بناء قاموس الحروف (Vocabulary) من جميع بيانات التدريب المولّدة
# ---------------------------------------------------------

def build_vocab_from_lists(list_paths):
    charset = set()
    for path in list_paths:
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) != 2:
                    continue
                _, label = parts
                charset.update(list(label))
    return sorted(list(charset))

all_train_lists = list(stage_list_paths.values()) + [val_list_path]
CHARSET = build_vocab_from_lists(all_train_lists)

# الفهرس 0 محجوز دائماً لرمز الـ CTC Blank
BLANK_IDX = 0
CHAR2IDX = {char: idx + 1 for idx, char in enumerate(CHARSET)}
IDX2CHAR = {idx + 1: char for idx, char in enumerate(CHARSET)}
NUM_CLASSES = len(CHARSET) + 1  # +1 من أجل رمز الـ Blank

print(f"✅ عدد الرموز الفريدة (الحروف/العلامات) المكتشفة: {len(CHARSET)}")
print(f"✅ إجمالي عدد الأصناف للنموذج (شامل Blank): {NUM_CLASSES}")

def encode_label(text):
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]

def decode_indices(indices):
    return "".join(IDX2CHAR.get(i, "") for i in indices)


# ---------------------------------------------------------
# 2) معمارية النموذج: CNN Feature Extractor + BiLSTM x2 + CTC Head
# ---------------------------------------------------------

class CNNFeatureExtractor(nn.Module):
    """
    مستخلص السمات البصرية (Convolutional Backbone) مبني بالكامل من الصفر.
    يقوم بتقليل ارتفاع الصورة تدريجياً مع الحفاظ على بُعد العرض،
    والذي سيمثل لاحقاً محور الزمن/التسلسل لطبقات BiLSTM.
    """
    def __init__(self, input_channels=1):
        super(CNNFeatureExtractor, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),            # H:64 -> 32

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),            # H:32 -> 16

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),  # H:16 -> 8

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),  # H:8 -> 4

            nn.Conv2d(512, 512, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),                            # H:4 -> 3
        )

    def forward(self, x):
        return self.cnn(x)  # (batch, 512, H', W')


class BiLSTMEncoder(nn.Module):
    """
    مشفّر تسلسلي ثنائي الاتجاه (Bidirectional LSTM) لنمذجة العلاقات
    السياقية بين الحروف/المقاطع المتتالية داخل السطر النصي.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(BiLSTMEncoder, self).__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, bidirectional=True, batch_first=True
        )
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        recurrent, _ = self.lstm(x)
        output = self.fc(recurrent)
        return output


class CustomThaiCRNN(nn.Module):
    """
    النموذج الكامل: CNN Extractor -> BiLSTM x2 -> Fully Connected -> CTC LogSoftmax
    مبني بالكامل من الصفر (Custom Architecture) دون أي أوزان مدربة مسبقاً.
    """
    def __init__(self, num_classes, hidden_size=256, cnn_output_channels=512):
        super(CustomThaiCRNN, self).__init__()
        self.cnn = CNNFeatureExtractor(input_channels=1)
        self.rnn1 = BiLSTMEncoder(cnn_output_channels, hidden_size, hidden_size)
        self.rnn2 = BiLSTMEncoder(hidden_size, hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, 1, H, W)
        conv = self.cnn(x)                        # (batch, C, H', W')
        batch, channels, height, width = conv.size()

        if height != 1:
            conv = nn.functional.adaptive_avg_pool2d(conv, (1, width))

        conv = conv.squeeze(2)                     # (batch, C, W)
        conv = conv.permute(0, 2, 1)                # (batch, W, C) -> تسلسل عبر محور العرض

        recurrent = self.rnn1(conv)
        recurrent = self.rnn2(recurrent)            # (batch, W, num_classes)

        output = recurrent.permute(1, 0, 2)         # (W, batch, num_classes) الشكل المطلوب لـ CTCLoss
        output = nn.functional.log_softmax(output, dim=2)
        return output


model = CustomThaiCRNN(num_classes=NUM_CLASSES).to(DEVICE)
print("\n✅ تم بناء النموذج CustomThaiCRNN بنجاح.")
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 إجمالي عدد المعاملات القابلة للتدريب: {total_params:,}")


### 🧠 الخلفية النظرية (الخلية 3) — CRNN + CTC

**لماذا CRNN (Convolutional Recurrent Neural Network)؟**
مهمة قراءة سطر نصي هي مزيج بين مسألتين: (1) مسألة بصرية بحتة (شكل كل حرف)، و(2) مسألة تسلسلية (ترتيب الحروف وسياقها). لذلك تُبنى المعمارية من جزأين متكاملين:

- **CNN Feature Extractor**: يستخرج سمات بصرية محلية من كل "شريحة عمودية" ضيقة من الصورة، ويقلّص الارتفاع تدريجياً إلى بُعد واحد بينما يحافظ على محور العرض — لأن هذا المحور سيمثّل تسلسل الزمن لاحقاً.
- **BiLSTM (Bidirectional LSTM)**: يقرأ هذا التسلسل من الاتجاهين (يسار→يمين ويمين→يسار)، وهو أمر بالغ الأهمية في اللغة التايلاندية لأن الحركات (Vowels) والعلامات الصوتية (Tone Marks) قد تظهر فوق أو تحت أو قبل الحرف الأساسي، فالسياق من الجهتين يساعد في حل الغموض البصري.

**لماذا CTC Loss (Connectionist Temporal Classification)؟**
لا نملك تحديد حدود كل حرف داخل الصورة (Character-level Alignment/Bounding Boxes) — فقط النص الكامل للسطر. دالة CTC تحل هذه المشكلة عبر تجميع كل المحاذاات (Alignments) الممكنة بين تسلسل الإطارات الناتج من الشبكة والنص الهدف، وحساب احتمالية النص الصحيح كمجموع كل هذه المحاذاات، دون الحاجة لتقطيع الصورة يدوياً حرفاً حرفاً.


In [ ]:
# ============================================================
# الخلية 4: التدريب التكاملي بأسلوب Curriculum Learning
#            (Auto-Resume + CTC Loss + tqdm Inline Progress)
# ============================================================

from PIL import Image
import torchvision.transforms as transforms
from tqdm.auto import tqdm

IMG_HEIGHT = 64
MAX_WIDTH = 256              # أقصى عرض للصورة بعد التحجيم (Resize + Padding)
BATCH_SIZE = 32
EPOCHS_PER_STAGE = 10
LEARNING_RATE = 1e-3
CHECKPOINT_PATH = os.path.join(CHECKPOINTS_DIR, "curriculum_checkpoint.pth")
SAVE_EVERY_N_BATCHES = 100


class ThaiOCRDataset(Dataset):
    def __init__(self, list_path, root_dir, img_height=IMG_HEIGHT, max_width=MAX_WIDTH):
        self.root_dir = root_dir
        self.img_height = img_height
        self.max_width = max_width
        self.samples = []
        with open(list_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) != 2:
                    continue
                rel_path, label = parts
                if len(encode_label(label)) == 0:
                    continue
                self.samples.append((rel_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rel_path, label = self.samples[idx]
        img_path = os.path.join(self.root_dir, rel_path)
        image = Image.open(img_path).convert("L")

        w, h = image.size
        new_w = max(1, int(w * (self.img_height / h)))
        new_w = min(new_w, self.max_width)
        image = image.resize((new_w, self.img_height), Image.BILINEAR)

        canvas = Image.new("L", (self.max_width, self.img_height), color=255)
        canvas.paste(image, (0, 0))

        tensor = transforms.ToTensor()(canvas)     # (1, H, W) بقيم بين 0 و 1
        tensor = (tensor - 0.5) / 0.5               # تطبيع بين -1 و 1

        target = torch.tensor(encode_label(label), dtype=torch.long)
        return tensor, target, new_w, len(target)


def ctc_collate_fn(batch):
    images, targets, input_lengths, target_lengths = zip(*batch)
    images = torch.stack(images, dim=0)
    targets_concat = torch.cat(targets)
    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return images, targets_concat, input_lengths, target_lengths


def get_dataloader_for_stage(stage_id):
    stage_name = STAGE_NAMES[stage_id]
    list_path = os.path.join(DATASET_DIR, stage_name, "train_list.txt")
    dataset = ThaiOCRDataset(list_path, root_dir=DATASET_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=ctc_collate_fn,
        num_workers=2,
        drop_last=True,
    )
    return loader


criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ---------------------------------------------------------
# آلية الاستئناف التلقائي (Auto-Resume)
# ---------------------------------------------------------
start_stage = 1
start_epoch = 1

if os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 تم العثور على نقطة حفظ سابقة: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_stage = checkpoint["stage"]
    start_epoch = checkpoint["epoch"] + 1
    if start_epoch > EPOCHS_PER_STAGE:
        start_stage += 1
        start_epoch = 1
    print(f"✅ تم استئناف التدريب تلقائياً من المرحلة {start_stage} - Epoch {start_epoch}")
else:
    print("🆕 لم يتم العثور على نقطة حفظ سابقة. سيبدأ التدريب من المرحلة 1.")

if start_stage > 5:
    print("🎉 التدريب على جميع المراحل الخمس مكتمل بالفعل!")
else:
    for stage_id in range(start_stage, 6):
        stage_name = STAGE_NAMES[stage_id]
        print(f"\n{'='*70}")
        print(f"🚀 بدء/استئناف المرحلة {stage_id}: {stage_name}")
        print(f"{'='*70}")

        train_loader = get_dataloader_for_stage(stage_id)
        epoch_range_start = start_epoch if stage_id == start_stage else 1

        for epoch in range(epoch_range_start, EPOCHS_PER_STAGE + 1):
            model.train()
            epoch_loss = 0.0
            num_batches = 0

            progress_bar = tqdm(
                train_loader,
                desc=f"المرحلة {stage_id}/5 | Epoch {epoch}/{EPOCHS_PER_STAGE}",
                leave=True,
            )

            for batch_idx, (images, targets, input_lengths, target_lengths) in enumerate(progress_bar):
                images = images.to(DEVICE)
                targets = targets.to(DEVICE)

                optimizer.zero_grad()
                log_probs = model(images)   # (W, batch, num_classes)

                seq_len = log_probs.size(0)
                capped_input_lengths = torch.clamp(input_lengths, max=seq_len)

                loss = criterion(log_probs, targets, capped_input_lengths, target_lengths)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

                batch_loss = loss.item()
                epoch_loss += batch_loss
                num_batches += 1

                progress_bar.set_postfix({
                    "Batch Loss": f"{batch_loss:.4f}",
                    "Epoch Avg Loss": f"{epoch_loss / num_batches:.4f}",
                })

                if (batch_idx + 1) % SAVE_EVERY_N_BATCHES == 0:
                    torch.save({
                        "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "stage": stage_id,
                        "epoch": epoch - 1,  # هذه الـ epoch لم تكتمل بعد
                    }, CHECKPOINT_PATH)

            avg_epoch_loss = epoch_loss / max(num_batches, 1)
            print(f"📉 نهاية Epoch {epoch}/{EPOCHS_PER_STAGE} للمرحلة {stage_id} | متوسط الخسارة: {avg_epoch_loss:.4f}")

            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "stage": stage_id,
                "epoch": epoch,
            }, CHECKPOINT_PATH)

        stage_final_path = os.path.join(CHECKPOINTS_DIR, f"model_stage{stage_id}_final.pth")
        torch.save(model.state_dict(), stage_final_path)
        print(f"💾 تم حفظ نموذج نهاية المرحلة {stage_id} في: {stage_final_path}")

        start_epoch = 1

    print("\n🎉🎉 اكتمل تدريب جميع مراحل الـ Curriculum Learning الخمس بنجاح! 🎉🎉")
    final_model_path = os.path.join(CHECKPOINTS_DIR, "model_final.pth")
    torch.save(model.state_dict(), final_model_path)
    print(f"💾 تم حفظ النموذج النهائي في: {final_model_path}")


### 🧠 الخلفية النظرية (الخلية 4) — حلقة التدريب التدرجي والاستئناف التلقائي

هذه الخلية تُترجم استراتيجية الـ Curriculum Learning إلى كود تنفيذي: حلقة خارجية تمر على المراحل الخمس بالترتيب (1→5)، وحلقة داخلية تُدرّب 10 Epochs لكل مرحلة باستخدام **نفس الأوزان** التي انتهت إليها المرحلة السابقة (لا يُعاد بناء النموذج من الصفر بين المراحل) — وهذا هو جوهر التدرج: النموذج "يبني" معرفته تراكمياً.

**آلية الاستئناف التلقائي (Auto-Resume):** بما أن جلسات Colab المجانية قد تنقطع فجأة، يتم حفظ حالة كاملة (أوزان النموذج، حالة المُحسّن Optimizer، رقم المرحلة، ورقم الـ Epoch) دورياً إلى Google Drive. عند إعادة تشغيل الخلية، يقرأ الكود آخر نقطة حفظ ويكمل التدريب من حيث توقف بالضبط، دون خسارة أي تقدّم.

**ملاحظة تستحق الإشارة إليها أمام لجنة المناقشة:** من المتوقع أن تظهر "قفزة" بسيطة في قيمة الخسارة (Loss) عند الانتقال من مرحلة إلى المرحلة الأصعب التالية (لأن توزيع البيانات تغيّر)، ثم تنخفض الخسارة تدريجياً من جديد — وهذا السلوك هو **الدليل العملي** على أن النموذج يتكيف تدريجياً مع كل مستوى صعوبة جديد، بدلاً من الانهيار الكامل الذي قد يحدث لو تم تدريبه على أصعب مرحلة مباشرة منذ البداية.


In [ ]:
# ============================================================
# الخلية 5: الاستنتاج (Inference) وتقييم نسبة خطأ الحروف (CER)
# ============================================================

import editdistance

def ctc_greedy_decode(log_probs):
    """
    فك تشفير Greedy: أخذ الصنف الأعلى احتمالاً في كل خطوة زمنية، ثم دمج
    التكرارات المتتالية وحذف رمز الـ Blank، وفق آلية عمل CTC القياسية.
    """
    # log_probs: (W, batch, num_classes)
    max_indices = torch.argmax(log_probs, dim=2)            # (W, batch)
    max_indices = max_indices.permute(1, 0).cpu().numpy()    # (batch, W)

    decoded_texts = []
    for sequence in max_indices:
        collapsed = []
        previous = None
        for idx in sequence:
            if idx != previous and idx != BLANK_IDX:
                collapsed.append(idx)
            previous = idx
        decoded_texts.append(decode_indices(collapsed))
    return decoded_texts


def load_trained_model(checkpoint_path=None):
    inference_model = CustomThaiCRNN(num_classes=NUM_CLASSES).to(DEVICE)
    path_to_load = checkpoint_path or os.path.join(CHECKPOINTS_DIR, "model_final.pth")

    if not os.path.exists(path_to_load):
        if os.path.exists(CHECKPOINT_PATH):
            print("⚠️ لم يتم العثور على النموذج النهائي، سيتم تحميل آخر نقطة حفظ متوفرة (Checkpoint).")
            checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
            inference_model.load_state_dict(checkpoint["model_state"])
        else:
            raise FileNotFoundError("❌ لا يوجد أي نموذج مدرّب أو نقطة حفظ لتحميلها بعد.")
    else:
        inference_model.load_state_dict(torch.load(path_to_load, map_location=DEVICE))

    inference_model.eval()
    return inference_model


def preprocess_image_for_inference(image_path, img_height=IMG_HEIGHT, max_width=MAX_WIDTH):
    image = Image.open(image_path).convert("L")
    w, h = image.size
    new_w = max(1, int(w * (img_height / h)))
    new_w = min(new_w, max_width)
    image = image.resize((new_w, img_height), Image.BILINEAR)

    canvas = Image.new("L", (max_width, img_height), color=255)
    canvas.paste(image, (0, 0))

    tensor = transforms.ToTensor()(canvas)
    tensor = (tensor - 0.5) / 0.5
    return tensor.unsqueeze(0)   # (1, 1, H, W)


def predict_text(image_path, inference_model):
    tensor = preprocess_image_for_inference(image_path).to(DEVICE)
    with torch.no_grad():
        log_probs = inference_model(tensor)
    predicted_text = ctc_greedy_decode(log_probs)[0]
    return predicted_text


def compute_cer(reference, hypothesis):
    """
    نسبة خطأ الحروف (Character Error Rate):
    CER = (Insertions + Deletions + Substitutions) / عدد حروف النص الصحيح
    تُحسب باستخدام مسافة تحرير ليفنشتاين (Levenshtein Edit Distance).
    """
    if len(reference) == 0:
        return 0.0 if len(hypothesis) == 0 else 1.0
    distance = editdistance.eval(reference, hypothesis)
    return distance / len(reference)


def evaluate_cer_on_validation_set(inference_model, val_list_path, root_dir=DATASET_DIR, max_samples=None):
    total_cer = 0.0
    total_samples = 0
    total_chars = 0
    total_edit_distance = 0

    with open(val_list_path, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n").split("\t") for line in f if len(line.strip()) > 0]

    if max_samples is not None:
        lines = lines[:max_samples]

    for rel_path, ground_truth in tqdm(lines, desc="📊 حساب CER على مجموعة التحقق"):
        img_path = os.path.join(root_dir, rel_path)
        predicted = predict_text(img_path, inference_model)

        distance = editdistance.eval(ground_truth, predicted)
        total_edit_distance += distance
        total_chars += len(ground_truth)
        total_cer += compute_cer(ground_truth, predicted)
        total_samples += 1

    average_cer_per_sample = total_cer / max(total_samples, 1)
    global_cer = total_edit_distance / max(total_chars, 1)

    print(f"\n📊 عدد عينات التقييم: {total_samples}")
    print(f"📊 متوسط CER لكل عينة: {average_cer_per_sample * 100:.2f}%")
    print(f"📊 نسبة CER الإجمالية (Global CER): {global_cer * 100:.2f}%")
    return global_cer


# ---------------------------------------------------------
# تشغيل التقييم الكامل + اختبار على صورة واحدة
# ---------------------------------------------------------

trained_model = load_trained_model()

print("🔍 تقييم النموذج على مجموعة التحقق (Validation Set)...")
global_cer_score = evaluate_cer_on_validation_set(trained_model, val_list_path)

with open(val_list_path, "r", encoding="utf-8") as f:
    sample_lines = f.readlines()

if len(sample_lines) > 0:
    sample_rel_path, sample_ground_truth = sample_lines[0].rstrip("\n").split("\t")
    sample_img_path = os.path.join(DATASET_DIR, sample_rel_path)

    predicted_sample = predict_text(sample_img_path, trained_model)

    print("\n🖼️ اختبار على صورة عينة:")
    print(f"   المسار          : {sample_img_path}")
    print(f"   النص الصحيح     : {sample_ground_truth}")
    print(f"   النص المتوقع    : {predicted_sample}")
    print(f"   CER لهذه العينة : {compute_cer(sample_ground_truth, predicted_sample) * 100:.2f}%")

    from IPython.display import display
    display(Image.open(sample_img_path))


### 🧠 الخلفية النظرية (الخلية 5) — فك التشفير وتقييم CER

**فك التشفير (Greedy CTC Decoding):** في كل خطوة زمنية عبر عرض الصورة، يختار النموذج الصنف (الحرف) الأعلى احتمالاً. نظراً لطبيعة CTC، قد يُكرّر النموذج نفس الحرف عدة خطوات متتالية لتمثيل حرف واحد ممتد بصرياً؛ لذا نُدمج التكرارات المتتالية ثم نحذف رمز الـ Blank للحصول على النص النهائي.

**لماذا CER وليس الدقة العادية (Accuracy)؟** في مهام التعرف على النصوص، مقارنة "تطابق كامل / لا تطابق" غير عادلة: توقّع نص طويل بحرف واحد خاطئ لا يساوي توقّع نص عشوائي بالكامل. لذلك نستخدم **نسبة خطأ الحروف (CER)**، المبنية على مسافة التحرير (Edit Distance)، والتي تقيس عدد عمليات الإضافة/الحذف/الاستبدال اللازمة لتحويل النص المتوقَّع إلى النص الصحيح، مقسومة على طول النص الصحيح — وهي المقياس القياسي المعتمد في أبحاث OCR لتقييم الأداء بدقة أعلى من الدقة الثنائية البسيطة.
